# Clase 5 — De modelado a evaluación: construye y valida un modelo

**Duración estimada: 120 minutos**

## Objetivos de aprendizaje

- Diferenciar modelos descriptivos y predictivos, y el rol del conjunto de entrenamiento.
- Entrenar un árbol de decisión con scikit-learn.
- Leer una matriz de confusión y calcular accuracy por clase.
- Entender qué mide la curva ROC y cómo ayuda a elegir el mejor modelo.

Este notebook es autocontenido: no necesitas haber ejecutado antes los notebooks de las clases 3 y 4. Empezamos recargando y limpiando el dataset de recetas desde cero, exactamente como quedó al final de la clase 4, y luego entrenamos y evaluamos un árbol de decisión real con scikit-learn.

## 1. Repaso rápido: cargar y limpiar el dataset de recetas

Antes de modelar necesitas datos limpios. Esto es un resumen del trabajo hecho en las clases 3 y 4: cargar el CSV, poner en minúsculas y normalizar los nombres de cocina, quitar cocinas con muy pocas recetas, y convertir "Yes"/"No" en 1/0.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import re
import random

# Descarga el CSV directamente desde el servidor de IBM y lo carga como DataFrame
recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

# La primera columna trae el nombre de la cocina, pero sin cabecera clara: la renombramos
column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names

# Normaliza a minúsculas para que "Mexican" y "mexican" no cuenten como cocinas distintas
recipes["cuisine"] = recipes["cuisine"].str.lower()

# Corrige nombres de cocina inconsistentes (el dataset original trae varias variantes)
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

# Elimina cocinas con menos de 50 recetas: no hay suficientes ejemplos para aprender un patrón fiable
recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

# Convierte las columnas de ingredientes de "Yes"/"No" a 1/0, formato que sklearn puede procesar
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

print("Dataset limpio y listo:", recipes.shape)

## 2. Modelado: descriptivo vs. predictivo

Antes de entrenar nada, recuerda la distinción del capítulo:

- Un **modelo descriptivo** examina relaciones entre variables para hacer inferencias ("si esto, entonces probablemente aquello").
- Un **modelo predictivo** produce un resultado de tipo sí/no (o, en nuestro caso, "¿de qué cocina es esta receta?").

Vamos a construir un modelo **predictivo**: dado un conjunto de ingredientes (1/0), predecir la cocina de origen. Para entrenarlo necesitamos un **conjunto de entrenamiento**: recetas de las que ya conocemos la cocina real, para que el algoritmo aprenda el patrón que las conecta con sus ingredientes.

### Caso CHF (recordatorio conceptual)

En el caso de la aseguradora de salud, el primer árbol de decisión para predecir reingreso de pacientes con insuficiencia cardíaca congestiva (CHF) tuvo un 85% de accuracy global, pero solo acertaba el 45% de los reingresos reales ("sí"). Ajustando el **costo relativo de clasificación errónea** (el peso que el algoritmo le da a equivocarse en el "sí" frente al "no"), el equipo probó tres configuraciones:

| Modelo | Costo relativo sí:no | Accuracy en "sí" | Accuracy en "no" | Accuracy global |
|---|---|---|---|---|
| 1 | 1:1 (default) | 45% | — | 85% |
| 2 | 9:1 | 97% | muy baja | 49% |
| 3 | 4:1 | 68% | 85% | 81% |

El modelo 3 fue el mejor balance. Vas a ver esta misma lógica de evaluación (accuracy por clase, no solo global) aplicada al dataset de recetas más abajo.

## 3. Entrenar un árbol de decisión con scikit-learn

Vamos a trabajar con un subconjunto de 5 cocinas: coreana, japonesa, china, tailandesa e india. `DecisionTreeClassifier` es la clase de scikit-learn que construye árboles de decisión para clasificación.

In [ ]:
from sklearn import tree

# Filtra el dataset limpio para quedarte solo con estas 5 cocinas
asian_indian_recipes = recipes[recipes.cuisine.isin(
    ["korean", "japanese", "chinese", "thai", "indian"]
)]

# La variable a predecir: la cocina de cada receta
cuisines = asian_indian_recipes["cuisine"]

# Las variables de entrada: todas las columnas de ingredientes
# (iloc[:, 1:] descarta la primera columna, que es "cuisine")
ingredients = asian_indian_recipes.iloc[:, 1:]

# Crea el árbol de decisión, limitando su profundidad a 3 niveles para que sea legible
bamboo_tree = tree.DecisionTreeClassifier(max_depth=3)

# .fit() entrena el modelo: le muestra ingredientes (entrada) + cocina real (salida)
# de cada receta, y el algoritmo aprende qué preguntas separan mejor las cocinas
bamboo_tree.fit(ingredients, cuisines)

print("Decision tree model saved to bamboo_tree!")

### Visualizar el árbol: de graphviz a plot_tree

El notebook original de IBM visualiza el árbol con la librería `graphviz`, instalada vía `!conda install python-graphviz --yes`. **Eso no funciona en Google Colab, porque Colab no tiene conda.** Por eso, en esta clase adaptamos el notebook original de IBM, que usaba graphviz vía conda, por `plot_tree` de scikit-learn, para que funcione en Colab sin instalar nada.

In [ ]:
import matplotlib.pyplot as plt

# plot_tree dibuja el árbol directamente con matplotlib: no necesita graphviz
plt.figure(figsize=(20, 10))
tree.plot_tree(
    bamboo_tree,
    feature_names=list(ingredients.columns.values),
    class_names=sorted(cuisines.unique()),
    filled=True,
    fontsize=8,
)
plt.show()

Leyendo el árbol puedes extraer reglas concretas, por ejemplo:

- Si una receta contiene **comino (cumin)** y **pescado (fish)**, pero NO **yogur (yoghurt)**, el modelo la clasifica como muy probablemente **tailandesa**.
- Si contiene **comino** pero NO **pescado** y NO **salsa de soja**, el modelo la clasifica como muy probablemente **india**.

Esa es la gran ventaja de un árbol de decisión frente a otros modelos: además de predecir, te deja "leer" el razonamiento detrás de cada predicción.

## 4. Evaluación: dividir en train/test

Entrenar y evaluar con los mismos datos sería hacer trampa: el modelo podría memorizar en vez de aprender un patrón general. Por eso separamos un **conjunto de prueba** (test), que el modelo nunca ve durante el entrenamiento, para comprobar si generaliza bien.

In [ ]:
bamboo = recipes[recipes.cuisine.isin(
    ["korean", "japanese", "chinese", "thai", "indian"]
)]

print(bamboo["cuisine"].value_counts())

In [ ]:
sample_n = 30
random.seed(1234)  # fija la semilla aleatoria para que el resultado sea reproducible

# Toma una muestra de 30 recetas de cada cocina para el conjunto de prueba
bamboo_test = bamboo.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))

bamboo_test_ingredients = bamboo_test.iloc[:, 1:]
bamboo_test_cuisines = bamboo_test["cuisine"]

# El conjunto de entrenamiento es todo lo que NO quedó en el conjunto de prueba
bamboo_test_index = bamboo.index.isin(bamboo_test.index)
bamboo_train = bamboo[~bamboo_test_index]

bamboo_train_ingredients = bamboo_train.iloc[:, 1:]
bamboo_train_cuisines = bamboo_train["cuisine"]

# Entrena un nuevo árbol, esta vez solo con el conjunto de entrenamiento
bamboo_train_tree = tree.DecisionTreeClassifier(max_depth=15)
bamboo_train_tree.fit(bamboo_train_ingredients, bamboo_train_cuisines)

print("Decision tree model saved to bamboo_train_tree!")

In [ ]:
# .predict() usa el modelo ya entrenado para predecir la cocina
# de recetas que el modelo NUNCA vio durante el entrenamiento
bamboo_pred_cuisines = bamboo_train_tree.predict(bamboo_test_ingredients)

bamboo_pred_cuisines[:10]

## 5. La matriz de confusión

La **matriz de confusión** es la herramienta central para leer qué tan bien predijo el modelo: las **filas representan la cocina real** y las **columnas representan la cocina que predijo el modelo**. Si el modelo fuera perfecto, todo el "peso" estaría en la diagonal (donde fila y columna coinciden). Convertimos cada fila a porcentaje, así que cada fila debería sumar 100%.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import itertools

test_cuisines = np.unique(bamboo_test_cuisines)
bamboo_confusion_matrix = confusion_matrix(bamboo_test_cuisines, bamboo_pred_cuisines, labels=test_cuisines)
title = "Bamboo Confusion Matrix"
cmap = plt.cm.Blues

plt.figure(figsize=(8, 6))
bamboo_confusion_matrix = (
    bamboo_confusion_matrix.astype("float") / bamboo_confusion_matrix.sum(axis=1)[:, np.newaxis]
) * 100

plt.imshow(bamboo_confusion_matrix, interpolation="nearest", cmap=cmap)
plt.title(title)
plt.colorbar()
tick_marks = np.arange(len(test_cuisines))
plt.xticks(tick_marks, test_cuisines)
plt.yticks(tick_marks, test_cuisines)

fmt = ".2f"
thresh = bamboo_confusion_matrix.max() / 2.0
for i, j in itertools.product(range(bamboo_confusion_matrix.shape[0]), range(bamboo_confusion_matrix.shape[1])):
    plt.text(
        j, i, format(bamboo_confusion_matrix[i, j], fmt),
        horizontalalignment="center",
        color="white" if bamboo_confusion_matrix[i, j] > thresh else "black",
    )

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

Con la semilla `1234`, una ejecución típica muestra que alrededor del **60% de las recetas chinas** se clasifican correctamente (un 37% se confunde con coreana, un 3% con india), y alrededor del **77% de las recetas indias** se clasifican correctamente. Como los árboles de decisión se construyen con muestreo aleatorio, tus números pueden variar ligeramente cada vez que ejecutes el código, pero el rendimiento general debería ser comparable.

Este es exactamente el mismo tipo de lectura que hicimos con la tabla del caso CHF: no basta con un número global de accuracy, hay que mirar clase por clase para saber dónde el modelo acierta y dónde se confunde.

In [ ]:
# Accuracy global: proporción de predicciones correctas sobre el total de recetas de prueba
accuracy_global = accuracy_score(bamboo_test_cuisines, bamboo_pred_cuisines)
print(f"Accuracy global del modelo: {accuracy_global:.2%}")

## 6. La curva ROC: cómo se elige el mejor modelo

La curva [ROC](../manual/01_glosario.md) (*receiver operating characteristic*) nació como herramienta de radar en la Segunda Guerra Mundial, para detectar aviones enemigos. Hoy es estándar en machine learning para comparar modelos de clasificación binaria.

Compara, para distintos valores de un criterio de discriminación (en el caso CHF, el costo relativo de clasificación errónea), la **tasa de verdaderos positivos** (sensibilidad) contra la **tasa de falsos positivos**. El modelo óptimo es el que logra la **máxima separación** entre su curva y la línea base de un modelo que adivina al azar.

En el caso CHF, de los 4 modelos evaluados con distintos costos relativos, el **modelo 3 (costo 4:1)** fue el que dio la máxima separación frente a la línea base — por eso fue el modelo elegido, no el que tenía el accuracy global más alto (modelo 1) ni el que mejor detectaba los "sí" (modelo 2).

Este notebook no construye una curva ROC con el dataset de recetas (es un problema multiclase con 5 cocinas, y la curva ROC clásica está pensada para clasificación binaria), pero la lógica es la misma que acabas de ver con la matriz de confusión: nunca te quedes con una sola métrica, compara el comportamiento del modelo en todas las clases y elige el que mejor equilibra los distintos tipos de error.

## Resumen

- Entrenaste un árbol de decisión (`DecisionTreeClassifier`) con `.fit()` sobre ingredientes de recetas para predecir su cocina.
- Visualizaste el árbol con `plot_tree` (en vez de graphviz, que no funciona en Colab) y leíste reglas concretas de clasificación.
- Dividiste los datos en entrenamiento y prueba, predijiste con `.predict()`, y leíste una matriz de confusión real (filas = real, columnas = predicho).
- Conectaste esta evaluación con el caso CHF: un accuracy global alto no garantiza un buen modelo si falla en la clase que más importa, y la curva ROC ayuda a elegir el modelo con mejor equilibrio.

Sigue con el notebook de práctica de esta clase para entrenar tu propio árbol con un subconjunto distinto de cocinas.